## Part 2: the language models

In [23]:
# Part 2 — Language Models

# 1. Configuration

# 2. Load tokenizers

## 3. Prepare LM datasets

## 4. Transformer architecture

## 5. Parameter counts

## 6. Training utilities

## 7. Sanity checks / tiny overfit

## 8. Character model training

## 9. BPE-3k model training

## 10. BPE-10k model training

## 11. Learning curves

## 12. Preliminary observations

In [24]:
# imports

from pathlib import Path
import json
import torch
import sentencepiece as spm

DATA_DIR = Path("/srv/data/lt2326-h26/a1")

PROJECT_DIR = Path.cwd().parent
TOKENIZER_DIR = PROJECT_DIR / "artifacts" / "tokenizers"
MODEL_DIR = PROJECT_DIR / "artifacts" / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A30


In [25]:
## Re-loading character tokenizer

with open(TOKENIZER_DIR / "char_vocab.json", encoding="utf-8") as f:
    char_vocab = json.load(f)

char_to_id = {char: i for i, char in enumerate(char_vocab)}

id_to_char = {i: char for i, char in enumerate(char_vocab)}

## Re-creating character encoder

def encode(text):
    token_ids = []

    for char in text:
        if char in char_to_id:
            token_id = char_to_id[char]
        else:
            token_id = char_to_id.get("<unk>", 0) # fallback for <unk>
        token_ids.append(token_id)

    return token_ids

print("Character vocab:", len(char_vocab))
print("UNK:", char_to_id["<unk>"])

Character vocab: 9443
UNK: 0


In [26]:
## Re-loading the BPE tokenizers

small_bpe = spm.SentencePieceProcessor(
    model_file=str(TOKENIZER_DIR / "small_bpe.model")
)

large_bpe = spm.SentencePieceProcessor(
    model_file=str(TOKENIZER_DIR / "large_bpe.model")
)

print("Character:", len(char_vocab))
print("Small BPE:", small_bpe.vocab_size())
print("Large BPE:", large_bpe.vocab_size())

text = "This is a test."

Character: 9443
Small BPE: 3000
Large BPE: 10000


In [27]:
## Encoding wrappers

def encode_char(text):
    return encode(text)

def encode_small_bpe(text):
    return small_bpe.encode(text, out_type=int)

def encode_large_bpe(text):
    return large_bpe.encode(text, out_type=int)

In [28]:
## Preparing to add eos to the vocab for each tokenizer ?????

# original vocab sizes
char_eos_id = len(char_vocab)
small_eos_id = small_bpe.vocab_size()
large_eos_id = large_bpe.vocab_size()

# vocab sizes with eos
char_lm_vocab_size = len(char_vocab) + 1
small_lm_vocab_size = small_bpe.vocab_size() + 1
large_lm_vocab_size = large_bpe.vocab_size() + 1

print("Character:", char_lm_vocab_size, "EOS:", char_eos_id)
print("BPE-3k:", small_lm_vocab_size, "EOS:", small_eos_id)
print("BPE-10k:", large_lm_vocab_size, "EOS:", large_eos_id)

Character: 9444 EOS: 9443
BPE-3k: 3001 EOS: 3000
BPE-10k: 10001 EOS: 10000


In [41]:
## Building token stream + adding eos to the vocab

def build_token_stream(data_path, encode_fn, eos_id):
    token_ids = [] # building one continuous list
    
    with open(data_path, encoding="utf-8") as f:
        for line in f:
            sentence = line.rstrip("\n")
            encoded_ids = encode_fn(sentence)

            token_ids.extend(encoded_ids)
            token_ids.append(eos_id)


    return token_ids

In [38]:
## shuffling data

# ... or not? if using balanced.txt instead ???

In [ ]:
# tokenizer conditions
tokenizers = {
    "Character": {
        "encode_fn": encode_char,
        "eos_id": char_eos_id,
        "vocab_size": char_lm_vocab_size,
    },
    "BPE-3k": {
        "encode_fn": encode_small_bpe,
        "eos_id": small_eos_id,
        "vocab_size": small_lm_vocab_size,
    },
    "BPE-10k": {
        "encode_fn": encode_large_bpe,
        "eos_id": large_eos_id,
        "vocab_size": large_lm_vocab_size,
    },
}

## building training streams for every tokenizer from balanced.txt

train_path = DATA_DIR / "tokenizer" / "balanced.txt"

train_streams = {}

for tokenizer_name, config in tokenizers.items():
    train_streams[tokenizer_name] = build_token_stream(
        data_path=train_path,
        encode_fn=config["encode_fn"],
        eos_id=config["eos_id"],
    )

Character 8980877
BPE-3k 6780586
BPE-10k 4221411


In [ ]:
# sanity checking training streams

for name in train_streams:
    train_streams[name] = torch.tensor(
        train_streams[name],
        dtype=torch.long
    )

for name, stream in train_streams.items():
    print(
        name,
        "tokens:", len(stream),
        "min ID:", stream.min().item(),
        "max ID:", stream.max().item(),
        "LM vocab size:", tokenizers[name]["vocab_size"],
    )

Character tokens: 8980877 min ID: 1 max ID: 9443 LM vocab size: 9444
BPE-3k tokens: 6780586 min ID: 0 max ID: 3000 LM vocab size: 3001
BPE-10k tokens: 4221411 min ID: 0 max ID: 10000 LM vocab size: 10001
